In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "VariableComparisons")
dataType = "AreaAverages"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Setup

# Region = "TRACER"; Case = "WET"; spinup_hours = "0"
# Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"
Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
# Region = "PRECIP"; Case = "DIURNAL"; spinup_hours = "12"

# Region = "Hawaii"; Case = "WET"; spinup_hours = "12"; spinup_hours="-16"
# Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

# RunType = (Region,Case,"TEMPO",spinup_hours)
# ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import RadarPlotting_Class

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = os.path.join(DirectoryManager.mainCodeDirectory, 'Functions_2.0')
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
####################################
#TESTING WEIGHTED AVERAGES

In [ ]:
def GetMean1(variable):
    variableMean = variable.mean(dim=("latitude","longitude"), skipna=True).data
    return variableMean

def GetMean2(variableSubset):
    #(1/A) times integral of phi dA 
    #dA is [(R*cos(Lat)dLon)][RdLat] = R^2 cos(Lat)dLatdLon ==> weight is simply cos(Lat)
    weights = np.cos(np.deg2rad(variableSubset.latitude))
    variableMean = variableSubset.weighted(weights).mean(
        dim=("latitude", "longitude"),
        skipna=True
    )
    return variableMean

In [ ]:
# a = ModelData.GetDataTimestep(t=100,varName='w')

# one = GetMean1(a)
# two = GetMean2(a)

# plt.plot(one,color='k')
# plt.plot(two,color='red')

In [ ]:
####################################
#TESTING VERTICAL INTERPOLATION

In [ ]:
#Functions

In [ ]:
def GetZGrids(ModelData):
    zGrid_f = ModelData.initData.zgrid
    zGrid_c = 0.5 * (
        zGrid_f.isel(nVertLevelsP1=slice(0, -1)) +
        zGrid_f.isel(nVertLevelsP1=slice(1, None))
    ); zGrid_c = zGrid_c.rename({"nVertLevelsP1": "nVertLevels"})

    return zGrid_f,zGrid_c

In [ ]:
a = GetZGrids(ModelData)[0].isel(latitude=-1,longitude=-1)
b = GetZGrids(ModelData)[0].isel(latitude=-1,longitude=-2)
print(a-b)
print(np.max(a-b)*1000)

In [ ]:
def GetZTarget(ModelData, zGrid_f,zGrid_c):
    """
    Return target vertical coordinates for both interface and center grids.
    """

    # -------------------------------
    # Interface grid (nVertLevelsP1)
    # -------------------------------
    zGrid0_f = zGrid_f.isel(nVertLevelsP1=0)

    idx_f = zGrid0_f.argmin(dim=("latitude", "longitude"))

    zTarget_f = zGrid_f.isel(
        latitude=idx_f["latitude"],
        longitude=idx_f["longitude"]
    ).data

    # -------------------------------
    # Center grid (nVertLevels)
    # -------------------------------
    zGrid0_c = zGrid_c.isel(nVertLevels=0)

    idx_c = zGrid0_c.argmin(dim=("latitude", "longitude"))

    zTarget_c = zGrid_c.isel(
        latitude=idx_c["latitude"],
        longitude=idx_c["longitude"]
    ).data

    return zTarget_f, zTarget_c

In [ ]:
def InterpolateVertical(variableSubset,zGrid_f,zGrid_c,zTarget):
    """
    Column-wise vertical interpolation to fixed height levels.
    """

    def interpColumn(varCol, zCol, zTarget):
        valid = np.isfinite(varCol) & np.isfinite(zCol)
        if valid.sum() < 2:
            return np.full(len(zTarget), np.nan)

        return np.interp(
            zTarget,
            zCol[valid],
            varCol[valid],
            left=np.nan,
            right=np.nan
        )

    if "nVertLevelsP1" in variableSubset.dims:
        zGrid = zGrid_f
        zDim = "nVertLevelsP1"
    elif "nVertLevels" in variableSubset.dims:
        zGrid = zGrid_c
        zDim = "nVertLevels"
        

    varInterp = xr.apply_ufunc(
        interpColumn,
        variableSubset,
        zGrid,
        input_core_dims=[[zDim], [zDim]],
        output_core_dims=[["z"]],
        vectorize=True,
        kwargs={"zTarget": zTarget},
        output_dtypes=[variableSubset.dtype],
    )

    varInterp = varInterp.assign_coords(
        z=("z", zTarget),
        latitude=variableSubset.latitude,
        longitude=variableSubset.longitude,
    )

    varInterp.name = variableSubset.name
    varInterp = varInterp.rename({"z": zDim})
    return varInterp

In [ ]:
def InterpolateVertical(variableSubset,zGrid_f,zGrid_c,zTarget_f,zTarget_c):
    """
    Column-wise vertical interpolation to fixed height levels.
    """

    # def interpColumn(varCol, zCol, zTarget):
    #     valid = np.isfinite(varCol) & np.isfinite(zCol)
    #     if valid.sum() < 2:
    #         return np.full(len(zTarget), np.nan)

    #     return np.interp(
    #         zTarget,
    #         zCol[valid],
    #         varCol[valid],
    #         left=np.nan,
    #         right=np.nan
    #     )

    def interpColumn(varCol, zCol, zTarget):
        return np.interp(
            zTarget,
            zCol,
            varCol,
            left=np.nan,
            right=np.nan
        )

    if "nVertLevelsP1" in variableSubset.dims:
        zGrid = zGrid_f
        zDim = "nVertLevelsP1"
        zTarget = zTarget_f
    elif "nVertLevels" in variableSubset.dims:
        zGrid = zGrid_c
        zDim = "nVertLevels"
        zTarget = zTarget_c
        

    varInterp = xr.apply_ufunc(
        interpColumn,
        variableSubset,
        zGrid,
        input_core_dims=[[zDim], [zDim]],
        output_core_dims=[["z"]],
        vectorize=True,
        kwargs={"zTarget": zTarget},
        output_dtypes=[variableSubset.dtype],
    )

    varInterp = varInterp.assign_coords(
        z=("z", zTarget),
        latitude=variableSubset.latitude,
        longitude=variableSubset.longitude,
    )

    varInterp.name = variableSubset.name
    varInterp = varInterp.rename({"z": zDim})
    return varInterp

In [ ]:
#1a: Testing with Subsetting near Large Terrain

In [ ]:
def FindMinHeightLocation(ModelData):
    terrainHeight = ModelData.initData.zgrid.isel(nVertLevelsP1=0)
    lat_index,lon_index = np.where(terrainHeight==terrainHeight.min())
    lon_location,lat_location = ModelData.longitude[lon_index].item(),ModelData.latitude[lat_index].item()
    return lon_location,lat_location 
def FindMaxHeightLocation(ModelData):
    terrainHeight = ModelData.initData.zgrid.isel(nVertLevelsP1=0)
    lat_index,lon_index = np.where(terrainHeight==terrainHeight.max())
    lon_location,lat_location = ModelData.longitude[lon_index].item(),ModelData.latitude[lat_index].item()
    return lon_location,lat_location 
def SubsetTaiwan(data):
    [lon_location,lat_location] = FindMaxHeightLocation(ModelData)
    distance=0.75
    return data.sel(longitude=slice(lon_location-distance,lon_location+distance),latitude=slice(lat_location-distance,lat_location+distance))

def GetZGrids_subset(ModelData):
    zGrid_f = ModelData.initData.zgrid
    zGrid_f = SubsetTaiwan(zGrid_f)
    zGrid_c = 0.5 * (
        zGrid_f.isel(nVertLevelsP1=slice(0, -1)) +
        zGrid_f.isel(nVertLevelsP1=slice(1, None))
    ); zGrid_c = zGrid_c.rename({"nVertLevelsP1": "nVertLevels"})

    return zGrid_f,zGrid_c

In [ ]:
terrainHeight = ModelData.initData.zgrid.isel(nVertLevelsP1=0)
terrainHeight = SubsetTaiwan(terrainHeight)
terrainHeight.plot()

In [ ]:
#1b: First testing for single column interpolation

In [ ]:
# varName = 'uReconstructZonal'
varName = 'w'
variableSubset = ModelData.GetDataTimestep(t=60,varName=varName)
variableSubset = SubsetTaiwan(variableSubset)

zGrid_f,zGrid_c = GetZGrids_subset(ModelData)

In [ ]:
[zTarget_f, zTarget_c] = GetZTarget(ModelData, zGrid_f, zGrid_c)

In [ ]:
plt.plot(zTarget_f,'k')
plt.plot(zTarget_c,'red')

In [ ]:
def interpColumn(varCol, zCol, zTarget):
    valid = np.isfinite(varCol) & np.isfinite(zCol)
    if valid.sum() < 2:
        return np.full(len(zTarget), np.nan)

    return np.interp(
        zTarget,
        zCol[valid],
        varCol[valid],
        left=np.nan,
        right=np.nan
    )

[lon_location,lat_location] = FindMaxHeightLocation(ModelData)
# [lon_location,lat_location] = FindMaxHeightLocation(ModelData)
varCol = variableSubset.sel(latitude=lat_location,longitude=lon_location)
zGrid_f,zGrid_c = GetZGrids_subset(ModelData)
zGrid = zGrid_f if varName in ['w'] else zGrid_c
zCol = zGrid.sel(latitude=lat_location,longitude=lon_location)

[zTarget_f, zTarget_c] = GetZTarget(ModelData, zGrid_f, zGrid_c)
zTarget = zTarget_f if varName in ['w'] else zTarget_c
varInterp = interpColumn(varCol, zCol, zTarget)

In [ ]:
#1c: testing over entire subsetting region

In [ ]:
zGrid_f,zGrid_c = GetZGrids_subset(ModelData)
[zTarget_f, zTarget_c] = GetZTarget(ModelData, zGrid_f, zGrid_c)
zTarget = zTarget_f if varName in ['w'] else zTarget_c
variableSubset_interpZ = InterpolateVertical(variableSubset,zGrid_f,zGrid_c,zTarget_f,zTarget_c)

In [ ]:
def TestPlot(variableSubset, variableSubset_interpZ, level):
    import numpy as np
    import matplotlib.pyplot as plt

    # ---- detect vertical dimension
    if "nVertLevelsP1" in variableSubset.dims:
        zDim = "nVertLevelsP1"
    elif "nVertLevels" in variableSubset.dims:
        zDim = "nVertLevels"
    else:
        raise ValueError("No recognized vertical dimension")

    # ---- compute joint min/max at this level
    vmin = np.nanmin([
        variableSubset.isel({zDim: level}).min().item(),
        variableSubset_interpZ.isel({zDim: level}).min().item()
    ])

    vmax = np.nanmax([
        variableSubset.isel({zDim: level}).max().item(),
        variableSubset_interpZ.isel({zDim: level}).max().item()
    ])

    levels = np.linspace(vmin, vmax, 20)

    # ---- difference (original - interpolated)
    diff = (
        variableSubset.isel({zDim: level})
        - variableSubset_interpZ.isel({zDim: level})
    )

    # ---- symmetric levels for difference
    diffMax = np.nanmax(np.abs(diff.values))
    diffLevels = np.linspace(-diffMax, diffMax, 20)

    fig, axes = plt.subplots(
        nrows=1, ncols=3, figsize=(18, 5), sharey=True
    )

    # Original
    variableSubset.isel({zDim: level}).plot(
        ax=axes[0],
        levels=levels
    )
    axes[0].set_title("Original")

    # Interpolated
    variableSubset_interpZ.isel({zDim: level}).plot(
        ax=axes[1],
        levels=levels
    )
    axes[1].set_title("Interpolated")

    # Difference
    diff.plot(
        ax=axes[2],
        levels=diffLevels,
        cmap="RdBu_r"
    )
    axes[2].set_title("Original − Interpolated")

    plt.tight_layout()
    plt.show()


In [ ]:
TestPlot(variableSubset,variableSubset_interpZ, level=15)
TestPlot(variableSubset,variableSubset_interpZ, level=23)
TestPlot(variableSubset,variableSubset_interpZ, level=50)

In [ ]:
mean1 = GetMean2(variableSubset)
mean2 = GetMean2(variableSubset_interpZ)
plt.plot(mean1,np.arange(len(mean1)),color='k')
plt.plot(mean2,np.arange(len(mean2)),color='red')

In [ ]:
#2: testing over entire domain

In [ ]:
varName = 'w'
# varName = 'qc'
variableSubset = ModelData.GetDataTimestep(t=60,varName=varName)

In [ ]:
# zGrid_f,zGrid_c = GetZGrids(ModelData)
zGrid_f,zGrid_c = ModelData.GetZGrids()
[zTarget_f, zTarget_c] = ModelData.GetZTarget(zGrid_f, zGrid_c)
variableSubset_interpZ = ModelData.InterpolateVertical(variableSubset,zGrid_f,zGrid_c,zTarget_f,zTarget_c)

In [ ]:
TestPlot(variableSubset,variableSubset_interpZ, level=10)
TestPlot(variableSubset,variableSubset_interpZ, level=15)
TestPlot(variableSubset,variableSubset_interpZ, level=53)

In [ ]:
mean1 = GetMean2(variableSubset)
mean2 = GetMean2(variableSubset_interpZ)
plt.plot(mean1,np.arange(len(mean1)),color='k')
plt.plot(mean2,np.arange(len(mean2)),color='red')

In [ ]:
#3: how to run

In [ ]:
# if any(dim.startswith("nVertLevels") for dim in variableSubset.dims):
#     if zTarget is None:
#         [zTarget_f, zTarget_c] = ModelData.GetZTarget(zGrid_f, zGrid_c)
#         zTarget = "loaded"
#     variableSubset = ModelData.InterpolateVertical(variableSubset,zGrid_f,zGrid_c,zTarget_f,zTarget_c)